In [2]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 37.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics]


In [3]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torchvision.models import vgg16, VGG16_Weights, alexnet, AlexNet_Weights
from torchmetrics.classification import Accuracy, F1Score
from tqdm.notebook import tqdm, trange

# ==========================================
# 1. Configuration & Reproductibilité
# ==========================================
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilisation du périphérique : {device}")

# ==========================================
# 2. Préparation des Données (MNIST RGB 224x224)
# ==========================================
# VGG16 et AlexNet s'attendent à des images RGB (3 canaux) de dimensions appropriées
transform_tl = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform_tl)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform_tl)

# DataLoaders (Batch size de 32 ou 64 recommandé sur GPU T4)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ==========================================
# 3. Définition des Modèles (Transfer Learning)
# ==========================================
def get_fine_tuned_vgg16(num_classes=10):
    model = vgg16(weights=VGG16_Weights.DEFAULT)
    # Remplacement de la dernière couche du classifieur pour 10 classes
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, num_classes)
    return model

def get_fine_tuned_alexnet(num_classes=10):
    model = alexnet(weights=AlexNet_Weights.DEFAULT)
    # Remplacement de la dernière couche du classifieur pour 10 classes
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, num_classes)
    return model

# ==========================================
# 4. Fonction d'Entraînement et d'Évaluation
# ==========================================
def train_and_evaluate_transfer_model(model_name, model, train_loader, test_loader, epochs=3, lr=0.0001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Initialisation des métriques
    test_acc_metric = Accuracy(task="multiclass", num_classes=10).to(device)
    test_f1_metric = F1Score(task="multiclass", num_classes=10, average='macro').to(device)

    print(f"\n==========================================")
    print(f" Début de l'entraînement : {model_name}")
    print(f"==========================================")

    start_train_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        # Progression dans l'époque
        for images, labels in tqdm(train_loader, desc=f"Époque {epoch+1}/{epochs}", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        print(f"Époque [{epoch+1}/{epochs}] - Train Loss: {epoch_train_loss:.4f}")

    training_time = time.time() - start_train_time
    print(f"Temps d'entraînement total : {training_time:.2f} s")

    # Évaluation sur le jeu de test
    model.eval()
    test_loss = 0.0
    test_acc_metric.reset()
    test_f1_metric.reset()

    start_eval_time = time.time()
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Évaluation Test", leave=False):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            test_acc_metric.update(preds, labels)
            test_f1_metric.update(preds, labels)

    final_test_loss = test_loss / len(test_loader.dataset)
    final_test_acc = test_acc_metric.compute().item()
    final_test_f1 = test_f1_metric.compute().item()
    eval_time = time.time() - start_eval_time

    # Affichage synthétique des résultats
    print(f"\n--- Résultats sur le Test ({model_name}) ---")
    print(f"Loss (Test)     : {final_test_loss:.4f}")
    print(f"Accuracy (Test) : {final_test_acc:.4f}")
    print(f"F1-Score (Test) : {final_test_f1:.4f}")
    print(f"Temps d'éval    : {eval_time:.2f} s")

    return {
        'Model': model_name,
        'Accuracy': final_test_acc,
        'F1_Score': final_test_f1,
        'Loss': final_test_loss,
        'Training_Time_s': training_time
    }

# ==========================================
# 5. Exécution pour VGG16 et AlexNet
# ==========================================

# --- Fine-Tuning AlexNet ---
alexnet_model = get_fine_tuned_alexnet(num_classes=10)
results_alexnet = train_and_evaluate_transfer_model(
    model_name="AlexNet (Fine-Tuned)",
    model=alexnet_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=0.0001
)

# --- Fine-Tuning VGG16 ---
vgg16_model = get_fine_tuned_vgg16(num_classes=10)
results_vgg16 = train_and_evaluate_transfer_model(
    model_name="VGG16 (Fine-Tuned)",
    model=vgg16_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=3,
    lr=0.0001
)

Utilisation du périphérique : cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.14MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 136kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.42MB/s]


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 161MB/s]



 Début de l'entraînement : AlexNet (Fine-Tuned)


Époque 1/3:   0%|          | 0/938 [00:00<?, ?it/s]

... (13 outputs left out)

Époque 3/3:   0%|          | 0/938 [00:00<?, ?it/s]

Époque [3/3] - Train Loss: 0.0203
Temps d'entraînement total : 3008.75 s


Évaluation Test:   0%|          | 0/157 [00:00<?, ?it/s]


--- Résultats sur le Test (VGG16 (Fine-Tuned)) ---
Loss (Test)     : 0.0173
Accuracy (Test) : 0.9950
F1-Score (Test) : 0.9950
Temps d'éval    : 68.40 s
